# GEO 371T/391: Climate Data - Spring 2026
# Assignment 12
# Future Changes in Cumulative Precipitation Events

*Developed by Cameron Cummins: 4/14/2026*

*Updated by Geeta Persad: 4/15/2026*

**In this notebook, we will leverage our full dataset from both CMIP6 and the GFDL Large Ensemble to understand how the cumulative rainfall metrics we've focused on in this class may change into the future in the face of scenario uncertainty, structural/model uncertainty, and internal variability.**


**You will have the following tasks in this assignment, some of which we will work on in lecture and some of which you will complete outside of class time in the Assignment 12 Template document on Canvas:**

- **Task 1:** Explore the future evolution of globally averaged cumulative rainfall metrics.
- **Task 1G: [Required for Grad Students; 1 pt Extra Credit for Undergrads]** Recreate Figure 1 for your group's region of interest.
- **Task 2:** Explore the geographic distribution of future trends in cumulative rainfall metrics across future scenarios.
- **Task 3:** Explore model diversity in the geographic distribution of future trends in 5-day cumulative rainfall to see how the "best" model compares to others.
- **Task 4:** Compare the magnitude of structural uncertainty versus internal variability in projecting future trends in 5-day cumulative rainfall.

## Step 1: Run the below code block

### This code block imports our packages and data and defines our mapping projection and our land mask.

In [ ]:
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib
import cartopy.io.shapereader as shpreader
from os import listdir
from shapely.geometry import shape
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import xarray as xr
matplotlib.style.use('fast')


class WinkelTripel(ccrs._WarpedRectangularProjection):
	"""
	Winkel-Tripel projection implementation for Cartopy
	"""

	def __init__(self, central_longitude=0.0, central_latitude=0.0, globe=None):
		globe = globe or ccrs.Globe(semimajor_axis=ccrs.WGS84_SEMIMAJOR_AXIS)
		proj4_params = [('proj', 'wintri'),
						('lon_0', central_longitude),
						('lat_0', central_latitude)]

		super(WinkelTripel, self).__init__(proj4_params, central_longitude, globe=globe)

	@property
	def threshold(self):
		return 1e4


def get_land_mask(lats, lons):
    lons = np.sort(((lons + 180) % 360) - 180)
    land_shp = shpreader.natural_earth(
        resolution='110m', category='physical', name='land'
    )
    reader = shpreader.Reader(land_shp)
    land_geoms = [(geom, 1) for geom in reader.geometries()]

    transform = from_bounds(
        lons.min(), lats.min(), lons.max(), lats.max(),
        len(lons), len(lats)
    )

    grid = rasterize(
        land_geoms,
        out_shape=(len(lats), len(lons)),
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=True,
    )
    grid = grid[::-1]

    mask = xr.DataArray(
        data=grid.astype(bool),
        dims=["lat", "lon"],
        coords=dict(lat=lats, lon=lons),
    )
    mask.name = "land_mask"
    n_roll = -(mask.lon < 0).sum().item()
    mask = mask.assign_coords(lon=(mask.lon % 360)).roll(
        lon=n_roll, roll_coords=True
    )
    return mask

head_input_dir = "/scratch/07644/oxygen/GEO371T-Climate-Data"

cmip6_hist_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/cmip6_hist_metrics_yearly.nc")
cmip6_land_mask = get_land_mask(cmip6_hist_yearly.lat.values, cmip6_hist_yearly.lon.values)
cmip6_hist_yearly = cmip6_hist_yearly.where(cmip6_land_mask)
cmip6_ssp245_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/cmip6_ssp245_metrics_yearly.nc").where(cmip6_land_mask)
cmip6_ssp370_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/cmip6_ssp370_metrics_yearly.nc").where(cmip6_land_mask)
cmip6_ssp585_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/cmip6_ssp585_metrics_yearly.nc").where(cmip6_land_mask)

cmip6_hist_yearly = cmip6_hist_yearly.assign_coords(time=np.arange(cmip6_hist_yearly.time.size) + 1920)
cmip6_ssp245_yearly = cmip6_ssp245_yearly.assign_coords(time=np.arange(cmip6_ssp245_yearly.time.size) + 2015)
cmip6_ssp370_yearly = cmip6_ssp370_yearly.assign_coords(time=np.arange(cmip6_ssp370_yearly.time.size) + 2015)
cmip6_ssp585_yearly = cmip6_ssp585_yearly.assign_coords(time=np.arange(cmip6_ssp585_yearly.time.size) + 2015)

gfdl_hist_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/gfdl-le_historical_metrics_interp_yearly.nc").where(cmip6_land_mask)
gfdl_ssp585_yearly = xr.open_dataset(f"{head_input_dir}/data/netcdf_files/gfdl-le_ssp585_metrics_interp_yearly.nc").where(cmip6_land_mask)
gfdl_hist_yearly = gfdl_hist_yearly.assign_coords(time=np.arange(gfdl_hist_yearly.time.size) + 1921)
gfdl_ssp585_yearly = gfdl_ssp585_yearly.assign_coords(time=np.arange(gfdl_ssp585_yearly.time.size) + 2015)

cmip6_hist_trends = cmip6_hist_yearly.polyfit(dim="time", deg=1)
cmip6_ssp245_trends = cmip6_ssp245_yearly.polyfit(dim="time", deg=1)
cmip6_ssp370_trends = cmip6_ssp370_yearly.polyfit(dim="time", deg=1)
cmip6_ssp585_trends = cmip6_ssp585_yearly.polyfit(dim="time", deg=1)
gfdl_ssp585_trends = gfdl_ssp585_yearly.polyfit(dim="time", deg=1)

cmip6_hist_yearly_ts = cmip6_hist_yearly.weighted(np.cos(np.deg2rad(cmip6_hist_yearly.lat))).mean(dim=["lat", "lon"])
cmip6_ssp245_yearly_ts = cmip6_ssp245_yearly.weighted(np.cos(np.deg2rad(cmip6_ssp245_yearly.lat))).mean(dim=["lat", "lon"]) - cmip6_hist_yearly_ts.mean(dim="time")
cmip6_ssp370_yearly_ts = cmip6_ssp370_yearly.weighted(np.cos(np.deg2rad(cmip6_ssp370_yearly.lat))).mean(dim=["lat", "lon"]) - cmip6_hist_yearly_ts.mean(dim="time")
cmip6_ssp585_yearly_ts = cmip6_ssp585_yearly.weighted(np.cos(np.deg2rad(cmip6_ssp585_yearly.lat))).mean(dim=["lat", "lon"]) - cmip6_hist_yearly_ts.mean(dim="time")
gfdl_hist_yearly_ts = gfdl_hist_yearly.weighted(np.cos(np.deg2rad(gfdl_hist_yearly.lat))).mean(dim=["lat", "lon"])
gfdl_ssp585_yearly_ts = gfdl_ssp585_yearly.weighted(np.cos(np.deg2rad(gfdl_ssp585_yearly.lat))).mean(dim=["lat", "lon"]) - gfdl_hist_yearly_ts.mean(dim="time")

## Task 1: Future evolution of maximum annual cumulative rainfall events

The below code block creates a set of figures showing the globally-averaged time evolution of the maximum value of one-day (left), three-day (middle), and five-day (right) cumulative rainfall in each year over the historical period (bottom) and as differences from the historical average for each of the future scenarios (top). 

Bold lines show the CMIP6 average (solid) or the GFDL Large Ensemble average (dashed). Transparent lines show the individual CMIP6 models. Shading around GFDL Large Ensemble average captures the GFDL ensemble range.

These figures show an incredible density of information, but it is difficult to distinguish signals or draw conclusions. This could be improved by changing line colors, transparency, thickness, or style to allow signals to be distinguished. 


### STEPS
1. Adjust the line styles using guidance in the code comments below to generate versions of this plot that are optimized for addressing the questions listed in the Assignment 12 template on Canvas. You may choose to generate multiple versions of the plot to address different questions.
2. Save out the desired version of the plot to include in your Assignment 12 template.
    



In [ ]:
%matplotlib inline
f, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(23, 12))

#This code block defines the style of the GFDL Ensemble Average and CMIP6 Multimodel lines
#linewidth : defines the thickness of the line. Increase it to create a thicker line.
#zorder : defines the order in which this line will appear on the graph relative to other lines being plotted.
#         Increase the zorder to put the line in front, and decrease it to move it to the back
mean_line_kwargs = dict(
    linewidth=2,
    zorder=4
)

#This code block defines the style of the individual CMIP6 model lines
#alpha : defines the transparency of the line from 1 (no transparency) to 0 (fully transparent)
#linewidth : defines the thickness of the line. Increase it to create a thicker line.
#zorder : defines the order in which this line will appear on the graph relative to other lines being plotted.
#         Increase the zorder to put the line in front, and decrease it to move it to the back
model_line_kwargs = dict(
    alpha=0.2,
    linewidth=2,
    zorder=2
)
axis_fz = 14

#In the below code block, wherever you see 'color=' is where the color for the corresponding line is set
#Look for the notes below to identify which color to modify for each line in the graph


for index, (ax, metric) in enumerate([(ax1, "one_day_pr"), (ax2, "three_day_pr"), (ax3, "five_day_pr"), (ax4, "one_day_pr"), (ax5, "three_day_pr"), (ax6, "five_day_pr")]):
    if index > 2: #Historical plots (bottom row)
        cmip6_hist_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="black", label="CMIP6", **mean_line_kwargs) #CMIP6 multi-model mean line
        for model in cmip6_hist_yearly_ts.model.values:
            cmip6_hist_yearly_ts[metric].sel(model=model).plot(ax=ax, color="black", **model_line_kwargs) #loops through individual CMIP6 model lines

        gfdl_hist_yearly_ts[metric].mean(dim=["member"]).plot(ax=ax, color="black", label="GFDL-LE", linestyle="--", **mean_line_kwargs) #GFDL ensemble average line
        ax.fill_between( #creates the shading of the GFDL ensemble range around the GFDL ensemble average line
            gfdl_hist_yearly_ts.time,
            gfdl_hist_yearly_ts[metric].min(dim=["member"]),
            gfdl_hist_yearly_ts[metric].max(dim=["member"]),
            color="black",
            alpha=0.1
        )
            
        ax.set_xlim(1920, 2014)
    else: #Future plots (top row)
        #CMIP6 average SSP 2-4.5
        cmip6_ssp245_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#f69320", label="SSP2-4.5", **mean_line_kwargs) 
        #CMIP6 average SSP 3-7.0
        cmip6_ssp370_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#df0000", label="SSP3-7.0", **mean_line_kwargs) 
        #CMIP6 average SSP 5-8.5
        cmip6_ssp585_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#980002", label="SSP5-8.5", **mean_line_kwargs) 

        #GFDL ensemble average SSP 5-8.5
        gfdl_ssp585_yearly_ts[metric].mean(dim=["member"]).plot(ax=ax, color="#980002", label="SSP5-8.5 (GFDL-LE)", linestyle="--", **mean_line_kwargs)
        ax.fill_between(#creates the shading of the GFDL ensemble range around the GFDL ensemble average line
            gfdl_ssp585_yearly_ts.time,
            gfdl_ssp585_yearly_ts[metric].min(dim=["member"]),
            gfdl_ssp585_yearly_ts[metric].max(dim=["member"]),
            color="#980002",
            alpha=0.1
        )
        
        for model in cmip6_ssp245_yearly_ts.model.values:
            #loops through individual CMIP6 model lines for SSP 2-4.5
            cmip6_ssp245_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#f69320", **model_line_kwargs)
            #loops through individual CMIP6 model lines for SSP 3-7.0
            cmip6_ssp370_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#df0000", **model_line_kwargs)
            #loops through individual CMIP6 model lines for SSP 5-8.5
            cmip6_ssp585_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#980002", **model_line_kwargs)
        ax.set_xlim(2015, 2100)
    ax.set_xlabel("Time (Year)", fontsize=axis_fz)
    ax.legend(loc="upper left")
    ax.grid(zorder=0)

ax1.set_ylabel(r"$\Delta$ 1-Day Precip. (mm/day)", fontsize=axis_fz)
ax2.set_ylabel(r"$\Delta$ 3-Day Precip. (mm/3day)", fontsize=axis_fz)
ax3.set_ylabel(r"$\Delta$ 5-Day Precip. (mm/5day)", fontsize=axis_fz)
ax4.set_ylabel("1-Day Precip. (mm/day)", fontsize=axis_fz)
ax5.set_ylabel("3-Day Precip. (mm/3day)", fontsize=axis_fz)
ax6.set_ylabel("5-Day Precip. (mm/5day)", fontsize=axis_fz)

fz = 16
ax1.set_title("1-Day Anomalies Relative to Historical", fontsize=fz)
ax2.set_title("3-Day Anomalies Relative to Historical", fontsize=fz)
ax3.set_title("5-Day Anomalies Relative to Historical", fontsize=fz)
ax4.set_title("1-Day Historical Baseline", fontsize=fz)
ax5.set_title("3-Day Historical Baseline", fontsize=fz)
ax6.set_title("5-Day Historical Baseline", fontsize=fz)

f.suptitle("Globally-Averaged, Max. Yearly Precipitation with Model Spread for CMIP6 Scenarios", fontsize=24);

## Task 1G: Regional evolution of cumulative rainfall events
### Required for Graduate Students; 1 pt. extra credit for Undergraduate Students

The above code generates a plot for the global average.

Using the below code blocks, generate the corresponding plot for your final project group's selected region.

To do this, you will need to:
1. In the first block of code below, adjust the `LAT_RANGE` and `LON_RANGE` values within the parentheses to match the latitude and longitude bounds of your region that your group has identified.
2. Run the first block of code below to generate regional versions of the time-series data
3. In the second block of code below, Replace the text between the quotation marks in `Region_Name` with the name of your group's selected region (e.g. "South Asia"). 
4. Run the second block of code below to generate the regional versions of the time-series plots
5. Adjust the line styles similar to how you did for Figure 1 to address the same questions, plus how the time evolution in your region of interest compares with the global average.

In [ ]:
# 1. Define your region of interest
# Note: Ensure these match the direction of your data (e.g., slice(60, 20) if lat is North to South)
LAT_RANGE = slice(20, 60) 
LON_RANGE = slice(-80, 0) 

def get_regional_ts(ds, lat_bounds, lon_bounds):
    """Subselects a region and returns the area-weighted mean time series."""
    subset = ds.sel(lat=lat_bounds, lon=lon_bounds)
    weights = np.cos(np.deg2rad(subset.lat))
    return subset.weighted(weights).mean(dim=["lat", "lon"])

# 2. Process Historical Base Periods
regional_cmip6_hist_yearly_ts = get_regional_ts(cmip6_hist_yearly, LAT_RANGE, LON_RANGE)
regional_gfdl_hist_yearly_ts  = get_regional_ts(gfdl_hist_yearly, LAT_RANGE, LON_RANGE)

# Cache the historical means for the anomaly calculation
regional_cmip6_ref_mean = regional_cmip6_hist_yearly_ts.mean(dim="time")
regional_gfdl_ref_mean  = regional_gfdl_hist_yearly_ts.mean(dim="time")

# 3. Process SSP Scenarios (Subtracting the historical baseline)
regional_cmip6_ssp245_yearly_ts = get_regional_ts(cmip6_ssp245_yearly, LAT_RANGE, LON_RANGE) - regional_cmip6_ref_mean
regional_cmip6_ssp370_yearly_ts = get_regional_ts(cmip6_ssp370_yearly, LAT_RANGE, LON_RANGE) - regional_cmip6_ref_mean
regional_cmip6_ssp585_yearly_ts = get_regional_ts(cmip6_ssp585_yearly, LAT_RANGE, LON_RANGE) - regional_cmip6_ref_mean

regional_gfdl_ssp585_yearly_ts  = get_regional_ts(gfdl_ssp585_yearly, LAT_RANGE, LON_RANGE) - regional_gfdl_ref_mean


In [ ]:
%matplotlib inline
f, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(23, 12))

Region_Name = "My Region"

#This code block defines the style of the GFDL Ensemble Average and CMIP6 Multimodel lines
#linewidth : defines the thickness of the line. Increase it to create a thicker line.
#zorder : defines the order in which this line will appear on the graph relative to other lines being plotted.
#         Increase the zorder to put the line in front, and decrease it to move it to the back
mean_line_kwargs = dict(
    linewidth=2,
    zorder=4
)

#This code block defines the style of the individual CMIP6 model lines
#alpha : defines the transparency of the line from 1 (no transparency) to 0 (fully transparent)
#linewidth : defines the thickness of the line. Increase it to create a thicker line.
#zorder : defines the order in which this line will appear on the graph relative to other lines being plotted.
#         Increase the zorder to put the line in front, and decrease it to move it to the back
model_line_kwargs = dict(
    alpha=0.2,
    linewidth=2,
    zorder=2
)
axis_fz = 14

for index, (ax, metric) in enumerate([(ax1, "one_day_pr"), (ax2, "three_day_pr"), (ax3, "five_day_pr"), (ax4, "one_day_pr"), (ax5, "three_day_pr"), (ax6, "five_day_pr")]):
    if index > 2:
        regional_cmip6_hist_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="black", label="CMIP6", **mean_line_kwargs)
        for model in regional_cmip6_hist_yearly_ts.model.values:
            regional_cmip6_hist_yearly_ts[metric].sel(model=model).plot(ax=ax, color="black", **model_line_kwargs)

        regional_gfdl_hist_yearly_ts[metric].mean(dim=["member"]).plot(ax=ax, color="black", label="GFDL-LE", linestyle="--", **mean_line_kwargs)
        ax.fill_between(
            regional_gfdl_hist_yearly_ts.time,
            regional_gfdl_hist_yearly_ts[metric].min(dim=["member"]),
            regional_gfdl_hist_yearly_ts[metric].max(dim=["member"]),
            color="black",
            alpha=0.1
        )
            
        ax.set_xlim(1920, 2014)
    else:
        regional_cmip6_ssp245_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#f69320", label="SSP2-4.5", **mean_line_kwargs)
        regional_cmip6_ssp370_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#df0000", label="SSP3-7.0", **mean_line_kwargs)
        regional_cmip6_ssp585_yearly_ts[metric].mean(dim="model").plot(ax=ax, color="#980002", label="SSP5-8.5", **mean_line_kwargs)

        regional_gfdl_ssp585_yearly_ts[metric].mean(dim=["member"]).plot(ax=ax, color="#980002", label="SSP5-8.5 (GFDL-LE)", linestyle="--", **mean_line_kwargs)
        ax.fill_between(
            regional_gfdl_ssp585_yearly_ts.time,
            regional_gfdl_ssp585_yearly_ts[metric].min(dim=["member"]),
            regional_gfdl_ssp585_yearly_ts[metric].max(dim=["member"]),
            color="#980002",
            alpha=0.1
        )
        
        for model in cmip6_ssp245_yearly_ts.model.values:
            regional_cmip6_ssp245_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#f69320", **model_line_kwargs)
            regional_cmip6_ssp370_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#df0000", **model_line_kwargs)
            regional_cmip6_ssp585_yearly_ts[metric].sel(model=model).plot(ax=ax, color="#980002", **model_line_kwargs)
        ax.set_xlim(2015, 2100)
    ax.set_xlabel("Time (Year)", fontsize=axis_fz)
    ax.legend(loc="upper left")
    ax.grid(zorder=0)

ax1.set_ylabel(r"$\Delta$ 1-Day Precip. (mm/day)", fontsize=axis_fz)
ax2.set_ylabel(r"$\Delta$ 3-Day Precip. (mm/3day)", fontsize=axis_fz)
ax3.set_ylabel(r"$\Delta$ 5-Day Precip. (mm/5day)", fontsize=axis_fz)
ax4.set_ylabel("1-Day Precip. (mm/day)", fontsize=axis_fz)
ax5.set_ylabel("3-Day Precip. (mm/3day)", fontsize=axis_fz)
ax6.set_ylabel("5-Day Precip. (mm/5day)", fontsize=axis_fz)

fz = 16
ax1.set_title("1-Day Anomalies Relative to Historical", fontsize=fz)
ax2.set_title("3-Day Anomalies Relative to Historical", fontsize=fz)
ax3.set_title("5-Day Anomalies Relative to Historical", fontsize=fz)
ax4.set_title("1-Day Historical Baseline", fontsize=fz)
ax5.set_title("3-Day Historical Baseline", fontsize=fz)
ax6.set_title("5-Day Historical Baseline", fontsize=fz)

f.suptitle(Region_Name+" Regionally-Averaged, Max. Yearly Precipitation with Model Spread for CMIP6 Scenarios", fontsize=24);

## Task 2: Mapping trends in maximum annual cumulative rainfall events

The below code block creates a map of the trends over the period from 2015-2100 in the annual maximum value of each of the three cumulative rainfall metrics for each of the three future scenarios in the CMIP6 database.

Steps:
1. Run the below block of code and save out the resulting figure to paste into your Assignment 12 template.
2. Address questions related to this figure in the Assignment 12 template.

In [ ]:
one_day_levels = np.arange(-0.6, 0.7, 0.1)*10
three_day_levels = np.arange(-0.6, 0.7, 0.1)*10
five_day_levels = np.arange(-0.6, 0.7, 0.1)*10
fz = 26
pad = 20

kwargs = dict(
    transform=ccrs.PlateCarree(), cmap="BrBG", center=0, extend='both'
)

cb_labels = [
    "mm/day", "mm/3day", "mm/5day"
]

f, axes = plt.subplots(3, 3, figsize=(30, 15), facecolor='w', subplot_kw=dict(projection=WinkelTripel()))

(10*cmip6_ssp245_trends["one_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[0,0], levels=one_day_levels, **kwargs)
(10*cmip6_ssp245_trends["three_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/3day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[0,1], levels=three_day_levels, **kwargs)
(10*cmip6_ssp245_trends["five_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[0,2], levels=five_day_levels, **kwargs)

(10*cmip6_ssp370_trends["one_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[1,0], levels=one_day_levels, **kwargs)
(10*cmip6_ssp370_trends["three_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/3day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[1,1], levels=three_day_levels, **kwargs)
(10*cmip6_ssp370_trends["five_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[1,2], levels=five_day_levels, **kwargs)

(10*cmip6_ssp585_trends["one_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[2,0], levels=one_day_levels, **kwargs)
(10*cmip6_ssp585_trends["three_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/3day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[2,1], levels=three_day_levels, **kwargs)
(10*cmip6_ssp585_trends["five_day_pr_polyfit_coefficients"]).rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").sel(degree=1).mean(dim="model").where(cmip6_land_mask).plot(ax=axes[2,2], levels=five_day_levels, **kwargs)

for row in axes:
    for ax in row:
        ax.coastlines()
        ax.set_title("")

axes[0, 0].set_title("1-Day Max. Yearly Precip.", fontsize=fz, pad=pad)
axes[0, 1].set_title("3-Day Max. Yearly Precip.", fontsize=fz, pad=pad)
axes[0, 2].set_title("5-Day Max. Yearly Precip.", fontsize=fz, pad=pad)
axes[0, 0].text(-0.08, 0.3, "SSP2-4.5", rotation=90, transform=axes[0, 0].transAxes, fontsize=fz)
axes[1, 0].text(-0.08, -0.9, "SSP3-7.0", rotation=90, transform=axes[0, 0].transAxes, fontsize=fz)
axes[2, 0].text(-0.08, -2.1, "SSP5-8.5", rotation=90, transform=axes[0, 0].transAxes, fontsize=fz)

f.suptitle("10-Year Model-Averaged Trends in Precipitation for CMIP6 Scenarios (2015-2100)", fontsize=30);

## Task 3: Exploring storylines of future change

The below figure shows maps of the trends over the period from 2015-2100 in the annual maximum value of 5-day cumulative rainfall in the SSP 5-8.5 scenario for each of the CMIP6 models. 

In class previously, we've discussed different approaches to determining which model is "best" at simulating cumulative rainfall metrics based on their behavior over the historical period relative to observations.

Steps:
1. Look back at your notes to identify a model that you previously chose as the "best" model in prior in-class discussions or assignment responses. 
2. Run the below code block and save out the resulting figure to the Assignment 12 template
3. Address the questions in the Assignment 12 template, focused on discussing what your "best" model predicts will happen in the future and how this compares with the CMIP6 average and with other CMIP6 models. 

In [ ]:
f, axes = plt.subplots(5, 4, figsize=(30, 25), facecolor='w', subplot_kw=dict(projection=WinkelTripel()))

kwargs = dict(
    transform=ccrs.PlateCarree(), cmap="BrBG", center=0, extend='both', levels=np.arange(-1, 1.1, 0.1)
)

index = 0
for row in axes:
    for ax in row:
        index += 1
        if index < cmip6_ssp585_trends.model.size:
            cmip6_ssp585_trends["five_day_pr_polyfit_coefficients"].rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").sel(degree=1, model=cmip6_ssp585_trends.model.values[index]).plot(ax=ax, **kwargs)
            ax.set_title(f"{cmip6_ssp585_trends.model.values[index]}")
            ax.coastlines()
        else:
            ax.remove()

f.suptitle("CMIP6 Model Diversity of 10-Year Trends in Yearly-Maximum 5-Day Precipitation for SSP5-8.5", fontsize=30, y=0.91);

# Task 4: Comparing structural and internal variability uncertainty in future cumulative rainfall metrics

The below code block generates a figure showing a map of the trends over the period from 2015-2100 in the annual maximum value of 5-day cumulative rainfall in the SSP 5-8.5 scenario for the CMIP6 Multi-Model Average (top left) and the GFDL Ensemble Average (top right). The bottom row shows the CMIP6 Multi-Model Range as a percentage of the CMIP6 Multi-Model Average (bottom left) and the GFDL Ensemble Range as a percentage of the GFDL Ensemble Average (bottom right).

Steps:
1. Adjust the colormaps based by replacing the `cmap` value below with other colormap abbreviations to find a combination that you like, following best practices and resources from Dr. Labe's lecture.
2. Save out the figure and paste into Assignment 12 template.
3. Interpret the figure to respond to the questions in the Assignment 12 template.

In [ ]:
cmip6_ssp585_trends
gfdl_ssp585_trends

f, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(30, 15), facecolor='w', subplot_kw=dict(projection=WinkelTripel()))

mean_kwargs = dict(
    transform=ccrs.PlateCarree(), cmap="bwr_r", extend='both', levels=np.arange(-0.6, 0.7, 0.1)*10
)

ratio_kwargs = dict(
    transform=ccrs.PlateCarree(), cmap="Reds", extend='both', levels=np.arange(0, 401, 25)
)

gfdl_mean = gfdl_ssp585_trends.sel(degree=1).mean(dim="member")
cmip6_mean = cmip6_ssp585_trends.sel(degree=1).mean(dim="model")
gfdl_range = gfdl_ssp585_trends.sel(degree=1).max(dim="member") - gfdl_ssp585_trends.sel(degree=1).min(dim="member")
cmip6_range = cmip6_ssp585_trends.sel(degree=1).max(dim="model") - cmip6_ssp585_trends.sel(degree=1).min(dim="model")

cmip6_ratio = (cmip6_range / abs(cmip6_mean)).where(abs(gfdl_mean) > 0.1)
gfdl_ratio = (gfdl_range / abs(gfdl_mean)).where(abs(gfdl_mean) > 0.1)

(cmip6_mean["five_day_pr_polyfit_coefficients"]*10).rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").plot(ax=ax1, **mean_kwargs)
(gfdl_mean["five_day_pr_polyfit_coefficients"]*10).rename(r"$\Delta$ mm/5day / $\Delta$ 10yr $^{-1}$").plot(ax=ax2, **mean_kwargs)

(cmip6_ratio["five_day_pr_polyfit_coefficients"]*100).rename("Range of Trends relative to Mean Trend (%)").plot(ax=ax3, **ratio_kwargs)
(gfdl_ratio["five_day_pr_polyfit_coefficients"]*100).rename("Range of Trends relative to Mean Trend (%)").plot(ax=ax4, **ratio_kwargs)

ax1.coastlines()
ax2.coastlines()
ax3.coastlines()
ax4.coastlines()

fz = 30
ax1.set_title("CMIP6 Model Ensemble", fontsize=fz, pad=20)
ax2.set_title("GFDL Large Ensemble", fontsize=fz, pad=20)
ax3.set_title(None)
ax4.set_title(None)

ax1.text(-0.08, 0.3, "Mean Trends", rotation=90, transform=ax1.transAxes, fontsize=fz)
ax3.text(-0.08, 0.2, "Relative Range", rotation=90, transform=ax3.transAxes, fontsize=fz)

f.suptitle("5-Day Yearly Max. Precipitation Trends over 2015-2100 for SSP5-8.5", fontsize=42, y=1);